# Financials Agent

The **Financials Agent** runs in parallel with the News Agent after the Planner.

**No LLM call** — pure data fetch via **Yahoo Finance RapidAPI** (`httpx`).

**Fetches:**
| Field | Source module |
|---|---|
| current_price | `price.regularMarketPrice` |
| pe_ratio / forward_pe | `summaryDetail` |
| revenue_growth / earnings_growth | `financialData` |
| market_cap | `price.marketCap` |
| 52_week_high / low | `summaryDetail` |
| beta / price_to_book | `summaryDetail` / `defaultKeyStatistics` |
| analyst_target_price | `financialData.targetMeanPrice` |
| sector / industry | `assetProfile` |

**Output state key:** `financials` (dict)

In [ ]:
import nbimporter
from dotenv import load_dotenv
from tools.yfinance_tool import get_stock_data

load_dotenv()


def financials_node(state: dict) -> dict:
    """
    Pull live financial metrics from Yahoo Finance via RapidAPI.
    No LLM — pure structured data fetch.
    Returns: financials (dict of metrics).
    """
    ticker = state["ticker"]
    print(f"[Financials] Pulling data for {ticker} via Yahoo Finance RapidAPI...")

    data = get_stock_data(ticker)

    price = data.get("current_price")
    pe    = data.get("pe_ratio")
    mcap  = data.get("market_cap")

    mcap_str = f"${mcap:,.0f}" if mcap else "N/A"
    print(f"[Financials] Price=${price} | P/E={pe} | MktCap={mcap_str}")

    return {"financials": data}


In [ ]:
if __name__ == "__main__":
    import json

    # --- Demo: pull NVDA financials ---
    result = financials_node({"ticker": "NVDA"})
    print(json.dumps(result["financials"], indent=2, default=str))
